# Check whether the model generated is structurally valid and correct

We now have generated the new model using `export_savedmodel.py`; but we of course cannot be sure just by that code to see whether the model made in ONNX is the same as the model we originally had in Keras. We begin by seeing whether the model is structurally valid.

In [1]:
import onnx
import onnxruntime as ort
model = onnx.load("das_model.onnx")
onnx.checker.check_model(model)
print("ONNX model is valid")

ONNX model is valid


In [2]:
sess = ort.InferenceSession("das_model.onnx", providers=["CPUExecutionProvider"])
print(sess.get_inputs()[0].name, sess.get_inputs()[0].shape, sess.get_inputs()[0].type)

input_1 ['unk__981', 8192, 1] tensor(float)


We now have verified that at least the model is structurally valid, now we go on to see whether the models descrribe the same. The most important rule of verifying this is we have to verify that the models have the same input tensors, same shape and are in inference mode.

In [3]:
import numpy as np
import tensorflow as tf
import onnxruntime as ort

np.random.seed(0)
x = np.random.randn(1, 8192, 1).astype(np.float32)

# ---------- TF SavedModel inference ----------
sm = tf.saved_model.load("das_savedmodel")
infer = sm.signatures["serving_default"]

# Inspect signature to get correct input key
print("SavedModel inputs:", list(infer.structured_input_signature[1].keys()))
print("SavedModel outputs:", list(infer.structured_outputs.keys()))

# Usually there is exactly one input; pick it
input_key = list(infer.structured_input_signature[1].keys())[0]

y_tf_dict = infer(**{input_key: tf.convert_to_tensor(x)})
# Usually one output; pick it
out_key = list(y_tf_dict.keys())[0]
y_tf = y_tf_dict[out_key].numpy()

print("TF SavedModel output:", y_tf.shape, y_tf.dtype)

# ---------- ONNX inference ----------
sess = ort.InferenceSession("das_model.onnx", providers=["CPUExecutionProvider"])
print("ONNX inputs:", [i.name for i in sess.get_inputs()])
print("ONNX outputs:", [o.name for o in sess.get_outputs()])

# Your ONNX input name is "input_1" from earlier
y_onnx = sess.run(None, {"input_1": x})[0]
print("ONNX output:", y_onnx.shape, y_onnx.dtype)

# ---------- Numeric comparison ----------
abs_diff = np.abs(y_tf - y_onnx)
print("max abs diff:", abs_diff.max())
print("mean abs diff:", abs_diff.mean())
print("max rel diff:", abs_diff.max() / (np.max(np.abs(y_tf)) + 1e-8))

SavedModel inputs: ['input_1']
SavedModel outputs: ['up_sampling1d']
TF SavedModel output: (1, 8192, 2) float32
ONNX inputs: ['input_1']
ONNX outputs: ['up_sampling1d']
ONNX output: (1, 8192, 2) float32
max abs diff: 1.1920929e-07
mean abs diff: 1.4173565e-08
max rel diff: 1.1935753537996864e-07


Now we check whether the outputs are numerically within the tolerances:

In [ ]:
for i, (a, b) in enumerate(zip(y_tf, y_onnx)):
    print(f"\nOutput {i}")
    print("shape keras:", a.shape)
    print("shape onnx:", b.shape)

    max_abs = np.max(np.abs(a - b))
    mean_abs = np.mean(np.abs(a - b))

    print("max (diff):", max_abs)
    print("mean (diff):", mean_abs)

    np.testing.assert_allclose(
        a, b,
        rtol=1e-4,
        atol=1e-5
    )



Output 0
shape keras: (1, 1024, 2)
shape onnx: (1, 1024, 2)
max (diff): 2.3841858e-07
mean (diff): 5.7858415e-08


The `max(diff)` value here is exactly one difference float32 ULP $2^{-22}$, which means the values are as good as perfectly the same. Just to make sure that they're close to the same layer-by-layer

In [ ]:
layer_names = [l.name for l in m.layers]

debug_model = tf.keras.Model(
    inputs=m.input,
    outputs=[l.output for l in m.layers]
)

keras_acts = debug_model(x, training=False)

ort_outputs = sess.run(
    [o.name for o in sess.get_outputs()],
    {"input_1": x}
)

In [ ]:
np.testing.assert_allclose(y_tf, y_onnx, rtol=1e-4, atol=1e-5)

This has no output, therefore we can conclude that the layers are the same, therefore we know the models are the same.